# Implementación de modelos supervisados (clasificación/regresión) con Scikit-learn

En este estudio se seleccionaron dos variables como target para los modelos supervisados, en función de su relevancia clínica y del tipo de problema que representan:

## Clasificación para Antecedentes personales de cáncer (Sí/No).

Antecedentes personales de cáncer (clasificación binaria):  

Se eligió esta variable porque responde a una pregunta clínica clave: ¿el paciente ha tenido o no un cáncer previamente? Esta información es categórica (Sí/No), lo que la convierte en un problema de clasificación supervisada. Su análisis permite identificar factores asociados a la presencia de antecedentes oncológicos y construir modelos predictivos que apoyen la evaluación de riesgo en nuevos pacientes.

Variable objetivo utilizada: Antecedentes personales de cáncer — columna binaria generada a partir de la respuesta original del paciente (mapea 'Sí' → 1; 'No' → 0).

In [10]:
import sys
from pathlib import Path

# Detectar la raíz del repo buscando la carpeta 'src' hacia arriba
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if not (repo_root / 'src').exists():
    # Fallback: intentar dos niveles arriba (útil al ejecutar desde results/reports)
    repo_root = Path.cwd().parents[2] if len(Path.cwd().parents) >= 3 else Path.cwd()
sys.path.append(str(repo_root))

from src.data_preprocessing import load_data, encode_categoricals, split_and_scale_classification
from src.model_training import train_and_predict_classifiers
from src.model_evaluation import classification_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. CARGAR DATOS (usar ruta absoluta dentro del repo)
dataset_path = repo_root / 'notebooks' / 'dataset_chile_cancer_piel.csv'
df = load_data(str(dataset_path))

# 2. DEFINIR LA ETIQUETA (TARGET)
y = df['Antecedentes personales de cáncer'].map({'Sí': 1, 'No': 0})

# 3. DEFINIR LAS CARACTERÍSTICAS (X) — eliminar columnas de fuga si existen
X = df.drop(['Antecedentes personales de cáncer', 'Cáncer familiar 1er grado (tipo)'], axis=1, errors='ignore')

# 4. PREPROCESAMIENTO
X = encode_categoricals(X, drop_first=True)

# 5. DIVISIÓN Y ESCALADO
X_train_scaled, X_test_scaled, y_train, y_test, scaler = split_and_scale_classification(
    X, y, test_size=0.2, random_state=42, stratify=True
)

# 6. MODELOS
models = {
    "Regresion_Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "Bosque_Aleatorio": RandomForestClassifier(random_state=42),
}

# 7. ENTRENAMIENTO Y PREDICCIÓN
trained, preds = train_and_predict_classifiers(models, X_train_scaled, X_test_scaled, y_train)

# 8. RESULTADOS
for name, y_pred in preds.items():
    metrics = classification_metrics(y_test, y_pred)
    print(f'--- Resultados para: {name} ---')
    print(metrics)

--- Resultados para: Regresion_Logistica ---
{'accuracy': 0.625, 'precision': 0.23076923076923078, 'recall': 0.04411764705882353, 'f1': 0.07407407407407407}
--- Resultados para: Bosque_Aleatorio ---
{'accuracy': 0.635, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


## Regresión para Tamaño máximo (cm).

Tamaño máximo de la lesión (regresión):  
Se seleccionó porque es un indicador cuantitativo clave en la evolución clínica. Al ser un valor continuo (expresado en centímetros), corresponde a un problema de regresión supervisada. Modelar esta variable permite estimar el tamaño esperado de una lesión en función de características del paciente y factores de riesgo.

In [9]:



from src.data_preprocessing import load_data, encode_categoricals, split_and_scale_regression
from src.model_training import train_and_predict_classifiers
from src.model_evaluation import regression_metrics
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


# Nota: esta celda asume que `df` ya está cargado (ver celda anterior).
y_reg = df["Tamaño máximo (cm)"]
X_reg = df.drop(["Tamaño máximo (cm)"], axis=1)
X_reg = encode_categoricals(X_reg, drop_first=True)

# División y escalado
X_train_r_scaled, X_test_r_scaled, y_train_r, y_test_r, scaler_r = split_and_scale_regression(X_reg, y_reg, test_size=0.2, random_state=42)

# Modelos
lin_reg = LinearRegression()
rf_reg = RandomForestRegressor(random_state=42)
models = {"LinearRegression": lin_reg, "RandomForestRegressor": rf_reg}

# Entrenamiento manual para regresores (mantener consistencia con el notebook original)
for name, model in models.items():
    model.fit(X_train_r_scaled, y_train_r)
    y_pred = model.predict(X_test_r_scaled)
    print(f'--- {name} ---')
    print(regression_metrics(y_test_r, y_pred))


--- LinearRegression ---
{'mae': 1.5730349551594276, 'mse': 3.3099087761447676, 'rmse': 1.8193154691105025, 'r2': -0.023554707459312096}
--- RandomForestRegressor ---
{'mae': 1.6122000000000003, 'mse': 3.3924099000000005, 'rmse': 1.8418495866926812, 'r2': -0.049067318048859665}


Se interpretan los resultados en el siguiente archivo executed_03_model_evaluation.ipynb